In [4]:
import pandas as pd
import numpy as np
import os

DATA_DIR    = "/content"
STATIC_CSV  = f"{DATA_DIR}/feature_vectors_static.csv"
DYNAMIC_CSV = f"{DATA_DIR}/feature_vectors_syscallsbinders_frequency_5_Cat.csv"

def peek_columns(path, encoding="latin-1"):
    """Reads ONLY the header (nrows=0) — loads zero data rows, so it cannot OOM."""
    cols = list(pd.read_csv(path, nrows=0, encoding=encoding).columns)
    print(f"{os.path.basename(path)}: {len(cols)} columns")
    return cols

if os.path.exists(STATIC_CSV):
    sc = peek_columns(STATIC_CSV)          # 50k cols, but header-only => a few MB, safe
    print("  static first 5:", sc[:5])
if os.path.exists(DYNAMIC_CSV):
    dc = peek_columns(DYNAMIC_CSV)
    print("  dynamic last 5:", dc[-5:])
if not os.path.exists(DYNAMIC_CSV):
    print("files not found yet — run the upload + unzip cells first")

feature_vectors_static.csv: 50621 columns
  static first 5: ['Unnamed: 0', "' a:targetActivity+AD0-'com.mplus.lib.ui.main.MainActivity' tools:targetApi+AD0-'25'+AD4AXA-n            +ADw-intent+AC0-filter+AD4AXA-n                +ADw-action a:name+AD0-'android.intent.action.MAIN'/+AD4AXA-n                +ADw-category a:name+AD0-'android.intent.category.LAUNCHER'/+AD4AXA-n            +ADw-/intent+AC0-filter+AD4AXA-n        +ADw-/activity+AC0-alias+AD4AXA-n", "Can't find launcher app through android.intent.category.HOME category and android.intent.action.MAIN action", '+ACI-More than one BroadcastReceiver that handles android.intent.action.MEDIA+AF8-BUTTON was found', "Please replace '+AFw-xe3+AFw-x80+AFw-x90+AFw-xe5+AFw-xba+AFw-x94+AFw-xe7+AFw-x94+AFw-xa8+AFw-xe5+AFw-x8c+AFw-x85+AFw-xe5+AFw-x90+AFw-x8d+AFw-xe3+AFw-x80+AFw-x91.intent.action.COCKROACH' with application's packageName for UmengService in AndroidManifest+ACE-"]
feature_vectors_syscallsbinders_frequency_5_Cat.csv: 471 columns

In [5]:
# Static and dynamic can only be merged if they share an ID column.
# Both reads are header-only => no memory risk on the 595 MB file.
ID_KEYS = ("hash", "sha256", "sha", "md5", "id", "name", "pkg", "package")

def id_cols(cols):
    return [c for c in cols if c.strip().lower() in ID_KEYS]

if os.path.exists(STATIC_CSV) and os.path.exists(DYNAMIC_CSV):
    s_cols = list(pd.read_csv(STATIC_CSV,  nrows=0, encoding="latin-1").columns)
    d_cols = list(pd.read_csv(DYNAMIC_CSV, nrows=0, encoding="latin-1").columns)
    s_ids, d_ids = id_cols(s_cols), id_cols(d_cols)
    shared = {c.lower() for c in s_ids} & {c.lower() for c in d_ids}

    print("static id-like cols :", s_ids or "NONE")
    print("dynamic id-like cols:", d_ids or "NONE")
    if shared:
        print("=> MERGE POSSIBLE on:", shared, "— chunked static loader is worth building")
    else:
        print("=> NO shared key. Static & dynamic can't be row-aligned.")
        print("   Dynamic-only training is the correct path, not a compromise.")
else:
    print("run upload + unzip first")

static id-like cols : ['package']
dynamic id-like cols: NONE
=> NO shared key. Static & dynamic can't be row-aligned.
   Dynamic-only training is the correct path, not a compromise.


In [6]:
from google.colab import files
uploaded = files.upload()  # a file picker will pop up - select CSV.zip from your downloads

In [7]:
import zipfile

# check the correct file - note the exact filename with the space
print("Valid zip:", zipfile.is_zipfile("/content/CSV (1).zip"))

if zipfile.is_zipfile("/content/CSV (1).zip"):
    !unzip -o "/content/CSV (1).zip" -d /content/CSVs_new/
    !ls -la /content/CSVs_new/

Valid zip: False


In [8]:
def load_dynamic(path=DYNAMIC_CSV):
    df = pd.read_csv(path, low_memory=False)
    num = df.select_dtypes("number").columns
    # syscall/binder values are small non-negative counts -> uint shrinks them ~4x
    df[num] = df[num].apply(pd.to_numeric, downcast="unsigned")
    return df

dynamic_df = load_dynamic()
df = dynamic_df                      # the name the training cell downstream uses
print("DYNAMIC shape:", df.shape)
print("mem (MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print(df["Class"].value_counts())

DYNAMIC shape: (11598, 471)
mem (MB): 7.3
Class
3    3904
4    2546
2    2100
5    1795
1    1253
Name: count, dtype: int64


In [14]:
"""
BOI Sentinel AI - Risk Scoring Model Trainer (FINAL - single-file version)
============================================================================
Built around feature_vectors_syscallsbinders_frequency_5_Cat.csv only.
This file has 470 behavior/syscall columns + a real 'Class' label column,
for 11,598 apps. No static file needed - avoids the RAM crash entirely.
"""

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import shap
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, accuracy_score

# =====================================================================
# CELL 1: Load
# =====================================================================
df = pd.read_csv("/content/feature_vectors_syscallsbinders_frequency_5_Cat.csv", low_memory=False)
print("Shape:", df.shape)

# Confirmed mapping from CICMalDroid's published category sizes
CLASS_MAP = {
    1: "Adware",
    2: "Banking",
    3: "SMS",
    4: "Riskware",
    5: "Benign",
}
df["CategoryName"] = df["Class"].map(CLASS_MAP)
print(df["CategoryName"].value_counts())


# =====================================================================
# CELL 2: Map 470 raw columns -> your 12-feature schema
# =====================================================================
FEATURE_NAMES = [
    "dangerous_perm_count",
    "suspicious_api_count",
    "yara_match_count",
    "obfuscation_detected",
    "dynamic_code_loading",
    "hardcoded_url_count",
    "malicious_ioc_count",
    "sms_intercepted",
    "accessibility_abuse",
    "c2_connection_count",
    "runtime_downloads",
    "ai_confidence",
]

# Keyword groups tuned to the composite-behavior column style you've shown
# (e.g. ACCESS_PERSONAL_INFO___, ALTER_PHONE_STATE___, FS_ACCESS____)
KEYWORD_MAP = {
    "dangerous_perm_count": ["ACCESS_PERSONAL_INFO", "DEVICE_ACCESS", "ALTER_PHONE_STATE"],
    "suspicious_api_count": ["ANTI_DEBUG", "EXECUTE", "CREATE_PROCESS", "CREATE_THREAD"],
    "dynamic_code_loading": ["CREATE_PROCESS", "EXECUTE", "load"],
    "hardcoded_url_count": ["NETWORK", "connect", "url", "http"],
    "sms_intercepted": ["SMS", "sendTextMessage", "SEND_SMS", "RECEIVE_SMS"],
    "accessibility_abuse": ["ACCESSIBILITY"],
    "c2_connection_count": ["NETWORK_ACCESS", "connect", "socket", "send", "recv"],
    "runtime_downloads": ["FS_ACCESS", "write", "CREATE_FOLDER", "download"],
}


def sum_matching_columns(df: pd.DataFrame, keywords: list) -> pd.Series:
    matched = [c for c in df.columns if any(k.lower() in c.lower() for k in keywords)]
    if not matched:
        print(f"  WARNING: no columns matched {keywords}")
        return pd.Series(0, index=df.index)
    print(f"  {keywords[0]}... matched {len(matched)} columns, e.g. {matched[:3]}")
    return df[matched].sum(axis=1)


print("\nBuilding features...")
features = pd.DataFrame()
for feat_name, keywords in KEYWORD_MAP.items():
    features[feat_name] = sum_matching_columns(df, keywords)

# Binary-ize the flag-style features (they should be 0/1, not raw counts)
features["dynamic_code_loading"] = (features["dynamic_code_loading"] > 0).astype(int)
features["sms_intercepted"] = (features["sms_intercepted"] > 0).astype(int)
features["accessibility_abuse"] = (features["accessibility_abuse"] > 0).astype(int)

# No equivalent in this dataset - these come from YOUR static-analysis/YARA/
# threat-intel/LLM services at inference time. Neutral placeholders for training.
features["yara_match_count"] = 0
features["obfuscation_detected"] = 0
features["malicious_ioc_count"] = 0
features["ai_confidence"] = 0.5

features = features[FEATURE_NAMES]  # enforce exact column order
print("\nFinal feature matrix shape:", features.shape)
print(features.describe())


# =====================================================================
# CELL 3: Build risk-score target from category labels
# =====================================================================
SEVERITY = {          # "how bad is this family", 0-100. A policy choice, not learned.
    "Benign":   5,
    "Riskware": 35,
    "Adware":   55,
    "SMS":      75,
    "Banking":  95,
}
CLASSES = ["Benign", "Riskware", "Adware", "SMS", "Banking"]
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
severity_vec = np.array([SEVERITY[c] for c in CLASSES], dtype=np.float32)

y = df["CategoryName"].map(class_to_idx).astype(int).values

# =====================================================================
# CELL 4: Train
# =====================================================================
X = features.astype(np.float32).values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

w_train = compute_sample_weight("balanced", y_train)

dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, feature_names=FEATURE_NAMES)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=FEATURE_NAMES)

params = {
    "objective": "multi:softprob",      # Changed to multi:softprob from reg:squarederror
    "num_class": len(CLASSES),          # Addition
    "max_depth": 6,                     # 4 -> 6
    "eta":       0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "mlogloss",          # changed from rmse to mlogloss
}

model = xgb.train(
    params, dtrain, num_boost_round=300,         #Increased rounds to 300
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds = 30, verbose_eval=25, #Added early Stopping
)

#preds = model.predict(dtest)
#rmse = np.sqrt(np.mean((preds - y_test) ** 2))
#print(f"\nTest RMSE (0-1 scale): {rmse:.4f}  ->  on 0-100 scale: {rmse * 100:.2f}")
# Changed :
#----------------------------------------------------------------------------------------
proba    = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))  # (n, 5)
pred_idx = proba.argmax(axis=1)
print(f"\nCategory accuracy: {accuracy_score(y_test, pred_idx):.2%}")
print(classification_report(y_test, pred_idx, target_names=CLASSES))

risk_score_test = proba @ severity_vec
print("risk score sample:", np.round(risk_score_test[:10], 1))

# optional: same 4-band view as before, for a like-for-like comparison
def score_to_band(s):
    if s < 15: return "Safe"
    if s < 45: return "Low Risk"
    if s < 65: return "Suspicious"
    return "Highly Malicious"
true_band = pd.Series(severity_vec[y_test]).apply(score_to_band)
pred_band = pd.Series(risk_score_test).apply(score_to_band)
print("Band-level accuracy:", (true_band.values == pred_band.values).mean())
#----------------------------------------------------------------------------------------

# =====================================================================
# CELL 5: SHAP sanity check
# =====================================================================
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(X_test[:300])
sv = np.asarray(sv) if not isinstance(sv, list) else np.stack(sv, axis=0)

# collapse every axis except the one matching len(FEATURE_NAMES) (the feature axis)
feat_axis = sv.shape.index(len(FEATURE_NAMES))
other_axes = tuple(a for a in range(sv.ndim) if a != feat_axis)
mean_abs_shap = np.abs(sv).mean(axis=other_axes)   # -> shape (12,), one value per feature

print("\nFeature importance (mean |SHAP value|):")
for name, val in sorted(zip(FEATURE_NAMES, mean_abs_shap), key=lambda x: -x[1]):
    print(f"  {name:25s} {val:.4f}")
# =====================================================================
# CELL 6: Save
# =====================================================================
import os
os.makedirs("/content/output", exist_ok=True)
MODEL_OUT_PATH = "/content/output/xgb_risk_model.pkl"
joblib.dump(model, MODEL_OUT_PATH)
print(f"\nSaved to {MODEL_OUT_PATH}")
print("Download it and place at: services/risk-scoring/models/xgb_risk_model.pkl")

from google.colab import files
files.download(MODEL_OUT_PATH)

Shape: (11598, 471)
CategoryName
SMS         3904
Riskware    2546
Banking     2100
Benign      1795
Adware      1253
Name: count, dtype: int64

Building features...
  ACCESS_PERSONAL_INFO... matched 3 columns, e.g. ['ACCESS_PERSONAL_INFO___', 'ALTER_PHONE_STATE___', 'DEVICE_ACCESS_____']
  ANTI_DEBUG... matched 4 columns, e.g. ['ANTI_DEBUG_____', 'CREATE_PROCESS`_____', 'CREATE_THREAD_____']
  CREATE_PROCESS... matched 2 columns, e.g. ['CREATE_PROCESS`_____', 'EXECUTE_____']
  NETWORK... matched 21 columns, e.g. ['NETWORK_ACCESS____', 'NETWORK_ACCESS()____', 'NETWORK_ACCESS(READ)____']
  SMS... matched 2 columns, e.g. ['SMS_SEND____', 'isImsSmsSupported']
  ACCESSIBILITY... matched 4 columns, e.g. ['addAccessibilityInteractionConnection', 'getEnabledAccessibilityServiceList', 'removeAccessibilityInteractionConnection']
  NETWORK_ACCESS... matched 24 columns, e.g. ['NETWORK_ACCESS____', 'NETWORK_ACCESS()____', 'NETWORK_ACCESS(READ)____']
  FS_ACCESS... matched 21 columns, e.g. ['CREATE

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>